# Library Management Portal
**Assignment 3: End-to-End Portal Design**

A console-based Library Management Portal written in Python. Librarians (admins) can add, search, delete books and generate reports. Members can search, issue and return books.

**Concepts used:** functions (`def`), loops (`for`, `while`), file handling (JSON + CSV), exception handling (`try / except / finally`).

**Data files:** `data/books.json`, `data/users.json`, `data/transactions.csv`

## 1. Imports and file paths

We import `json` and `csv` for storage, `os` for file paths and `datetime` for dates in the log. The three file paths are kept as constants at the top so they are easy to change.

In [1]:
"""
Library Management Portal
-------------------------
A console-based portal to manage books, members, issuing and returning.
Data is stored in:
    data/books.json         -> all books in the library
    data/users.json         -> login accounts (admin / member)
    data/transactions.csv   -> log of every issue / return (appended)
"""

import json
import csv
import os
from datetime import datetime

DATA_FOLDER = "data"
BOOKS_FILE = os.path.join(DATA_FOLDER, "books.json")
USERS_FILE = os.path.join(DATA_FOLDER, "users.json")
LOG_FILE = os.path.join(DATA_FOLDER, "transactions.csv")

## 2. File handling functions

These functions are the only parts of the program that touch the files.

- `load_json()` reads a JSON file when the program starts. It uses **try / except / finally**: a missing file (`FileNotFoundError`), a blocked file (`PermissionError`) or a broken file (`JSONDecodeError`) returns default data instead of crashing, and `finally` always closes the file.
- `save_json()` writes the data back after every change.
- `log_transaction()` **appends** one line to a CSV file every time a book is issued or returned, so the history is never overwritten.

In [2]:
def load_json(file_path, default_data):
    """
    Read a JSON file and return its data.
    If the file is missing or broken, return default_data instead of crashing.
    """
    file = None
    try:
        file = open(file_path, "r")
        data = json.load(file)
        print(f"[OK] Loaded {len(data)} records from {file_path}")
        return data
    except FileNotFoundError:
        print(f"[WARNING] {file_path} not found. Starting with empty data.")
        return default_data
    except PermissionError:
        print(f"[ERROR] No permission to read {file_path}.")
        return default_data
    except json.JSONDecodeError:
        print(f"[ERROR] {file_path} is corrupted. Starting with empty data.")
        return default_data
    finally:
        if file is not None:
            file.close()


def save_json(file_path, data):
    """Write data to a JSON file. Returns True if saved, False if it failed."""
    file = None
    try:
        os.makedirs(DATA_FOLDER, exist_ok=True)
        file = open(file_path, "w")
        json.dump(data, file, indent=4)
        return True
    except PermissionError:
        print(f"[ERROR] No permission to write {file_path}.")
        return False
    except OSError as error:
        print(f"[ERROR] Could not save {file_path}: {error}")
        return False
    finally:
        if file is not None:
            file.close()


def log_transaction(action, book_id, username):
    """Append one line (date, action, book, user) to the CSV log file."""
    try:
        os.makedirs(DATA_FOLDER, exist_ok=True)
        new_file = not os.path.exists(LOG_FILE)
        with open(LOG_FILE, "a", newline="") as file:
            writer = csv.writer(file)
            if new_file:
                writer.writerow(["date_time", "action", "book_id", "username"])
            writer.writerow([datetime.now().strftime("%Y-%m-%d %H:%M"),
                             action, book_id, username])
        return True
    except (PermissionError, OSError) as error:
        print(f"[ERROR] Could not write log: {error}")
        return False


def read_transactions():
    """Read the CSV log and return a list of rows (dictionaries)."""
    try:
        with open(LOG_FILE, "r") as file:
            return list(csv.DictReader(file))
    except FileNotFoundError:
        return []

## 3. Login function

`user_login()` loops through the users list with a `for` loop and returns the matching user, or `None` if the username and password don't match. The user's **role** (admin or member) decides which menu they see.

In [3]:
def user_login(users, username, password):
    """
    Check username and password against the users list.
    Returns the matching user dictionary, or None if login fails.
    """
    for user in users:
        if user["username"] == username.strip() and user["password"] == password:
            return user
    return None

## 4. Add, search and delete book records

- `add_record()` converts year and copies with `int()` inside a `try` block. Typing text like `"abc"` raises a `ValueError`, which is caught and turned into a friendly message. It also checks for empty fields, invalid years and duplicate IDs.
- `search_record()` does a case-insensitive partial match on title, author or book ID with a `for` loop.
- `delete_record()` refuses to delete a book that still has copies issued.

Every function returns `(True/False, message)` so the menu can print the result and decide whether to save.

In [4]:
def add_record(books, book_id, title, author, year, copies):
    """
    Add a new book to the books list.
    year and copies may arrive as text, so they are converted to int here.
    Returns (True, message) on success or (False, message) on failure.
    """
    try:
        year = int(year)
        copies = int(copies)
    except ValueError:
        return False, "Year and copies must be whole numbers."

    book_id = book_id.strip().upper()
    if book_id == "" or title.strip() == "" or author.strip() == "":
        return False, "Book ID, title and author cannot be empty."
    if copies <= 0:
        return False, "Copies must be at least 1."
    if year < 1000 or year > datetime.now().year:
        return False, "Please enter a valid publication year."

    for book in books:
        if book["book_id"] == book_id:
            return False, f"A book with ID {book_id} already exists."

    new_book = {
        "book_id": book_id,
        "title": title.strip(),
        "author": author.strip(),
        "year": year,
        "total_copies": copies,
        "available": copies,
        "issued_to": []
    }
    books.append(new_book)
    return True, f"Book '{new_book['title']}' added successfully."


def search_record(books, keyword, field="title"):
    """
    Search books by 'title', 'author' or 'book_id'.
    Returns a list of matching books (case-insensitive partial match).
    """
    if field not in ("title", "author", "book_id"):
        return []
    keyword = keyword.strip().lower()
    results = []
    for book in books:
        if keyword in str(book[field]).lower():
            results.append(book)
    return results


def find_book(books, book_id):
    """Return the book with this exact ID, or None."""
    for book in books:
        if book["book_id"] == book_id.strip().upper():
            return book
    return None


def delete_record(books, book_id):
    """Remove a book if none of its copies are issued. Returns (bool, message)."""
    book = find_book(books, book_id)
    if book is None:
        return False, "Book not found."
    if len(book["issued_to"]) > 0:
        return False, "Cannot delete: some copies are still issued."
    books.remove(book)
    return True, f"Book '{book['title']}' deleted."

## 5. Issue and return books

`issue_book()` reduces `available` by 1 and records the member's username. `return_book()` does the opposite. Both check for invalid cases first (book not found, no copies left, not borrowed by this user) and write to the CSV log.

In [5]:
def issue_book(books, book_id, username):
    """Issue one copy of a book to a member. Returns (bool, message)."""
    book = find_book(books, book_id)
    if book is None:
        return False, "Book not found."
    if book["available"] <= 0:
        return False, f"No copies of '{book['title']}' are available."
    if username in book["issued_to"]:
        return False, "You already have this book."
    book["available"] -= 1
    book["issued_to"].append(username)
    log_transaction("ISSUE", book["book_id"], username)
    return True, f"'{book['title']}' issued to {username}."


def return_book(books, book_id, username):
    """Return a book that the member has borrowed. Returns (bool, message)."""
    book = find_book(books, book_id)
    if book is None:
        return False, "Book not found."
    if username not in book["issued_to"]:
        return False, "This book was not issued to you."
    book["available"] += 1
    book["issued_to"].remove(username)
    log_transaction("RETURN", book["book_id"], username)
    return True, f"'{book['title']}' returned by {username}."

## 6. Display and report functions

`display_books()` prints a formatted table with a `for` loop. `generate_report()` loops through all books and all logged transactions to calculate totals, out-of-stock titles and the most issued book. It **returns a dictionary**, which `print_report()` then displays.

In [6]:
def display_books(books):
    """Print books in a neat table using a for loop."""
    if len(books) == 0:
        print("No books to show.")
        return
    print(f"{'ID':<6} {'Title':<32} {'Author':<22} {'Year':<6} {'Avail':<6}")
    print("-" * 76)
    for book in books:
        print(f"{book['book_id']:<6} {book['title'][:31]:<32} "
              f"{book['author'][:21]:<22} {book['year']:<6} "
              f"{book['available']}/{book['total_copies']}")


def generate_report(books):
    """
    Build summary statistics about the library.
    Returns a dictionary so it can be printed or tested.
    """
    total_titles = len(books)
    total_copies = 0
    issued_copies = 0
    out_of_stock = []
    for book in books:
        total_copies += book["total_copies"]
        issued_copies += book["total_copies"] - book["available"]
        if book["available"] == 0:
            out_of_stock.append(book["title"])

    transactions = read_transactions()
    issue_count = {}
    for row in transactions:
        if row["action"] == "ISSUE":
            issue_count[row["book_id"]] = issue_count.get(row["book_id"], 0) + 1

    most_popular = None
    if issue_count:
        top_id = max(issue_count, key=issue_count.get)
        top_book = find_book(books, top_id)
        name = top_book["title"] if top_book else top_id
        most_popular = f"{name} ({issue_count[top_id]} issues)"

    return {
        "total_titles": total_titles,
        "total_copies": total_copies,
        "issued_copies": issued_copies,
        "available_copies": total_copies - issued_copies,
        "out_of_stock": out_of_stock,
        "total_transactions": len(transactions),
        "most_popular": most_popular
    }


def print_report(report):
    """Print the report dictionary in a readable way."""
    print("\n========== LIBRARY REPORT ==========")
    print(f"Total titles       : {report['total_titles']}")
    print(f"Total copies       : {report['total_copies']}")
    print(f"Copies issued      : {report['issued_copies']}")
    print(f"Copies available   : {report['available_copies']}")
    print(f"Out of stock       : {', '.join(report['out_of_stock']) or 'None'}")
    print(f"Total transactions : {report['total_transactions']}")
    print(f"Most popular book  : {report['most_popular'] or 'No issues yet'}")
    print("====================================")

## 7. Safe input helpers

`get_int_input()` uses a `while True` loop with `try / except ValueError`. It keeps asking until the user types a valid number in range, so wrong input never crashes the program.

In [7]:
def get_int_input(prompt, low, high):
    """Keep asking until the user types a whole number between low and high."""
    while True:
        try:
            value = int(input(prompt))
            if low <= value <= high:
                return value
            print(f"Please enter a number from {low} to {high}.")
        except ValueError:
            print("Invalid input. Please type a number.")


def get_text_input(prompt):
    """Keep asking until the user types something that is not empty."""
    while True:
        text = input(prompt).strip()
        if text != "":
            return text
        print("This field cannot be empty.")

## 8. Menus and main loop

`main()` loads the data from the files on startup, then runs the **main `while True` loop** until the user chooses Exit. After login, the user goes to the admin or member menu, and each menu is also a `while True` loop. Data is saved after each change and again on exit.

In [8]:
def login_screen(users):
    """Give the user 3 attempts to log in. Returns the user or None."""
    attempts = 3
    while attempts > 0:
        username = get_text_input("Username: ")
        password = get_text_input("Password: ")
        user = user_login(users, username, password)
        if user is not None:
            print(f"\nWelcome, {user['name']}! (role: {user['role']})")
            return user
        attempts -= 1
        print(f"Wrong username or password. {attempts} attempt(s) left.")
    print("Too many failed attempts.")
    return None


def admin_menu(books, user):
    """Menu loop for librarians/admins."""
    while True:
        print("\n--- ADMIN MENU ---")
        print("1. View all books\n2. Add book\n3. Search books\n"
              "4. Delete book\n5. Generate report\n6. Logout")
        choice = get_int_input("Choose (1-6): ", 1, 6)

        if choice == 1:
            display_books(books)
        elif choice == 2:
            ok, msg = add_record(books,
                                 get_text_input("Book ID: "),
                                 get_text_input("Title: "),
                                 get_text_input("Author: "),
                                 get_text_input("Year: "),
                                 get_text_input("Number of copies: "))
            print(msg)
            if ok:
                save_json(BOOKS_FILE, books)
        elif choice == 3:
            search_menu(books)
        elif choice == 4:
            ok, msg = delete_record(books, get_text_input("Book ID to delete: "))
            print(msg)
            if ok:
                save_json(BOOKS_FILE, books)
        elif choice == 5:
            print_report(generate_report(books))
        elif choice == 6:
            print("Logged out.")
            return


def member_menu(books, user):
    """Menu loop for library members (students)."""
    while True:
        print("\n--- MEMBER MENU ---")
        print("1. View all books\n2. Search books\n3. Issue book\n"
              "4. Return book\n5. My books\n6. Logout")
        choice = get_int_input("Choose (1-6): ", 1, 6)

        if choice == 1:
            display_books(books)
        elif choice == 2:
            search_menu(books)
        elif choice == 3:
            ok, msg = issue_book(books, get_text_input("Book ID: "), user["username"])
            print(msg)
            if ok:
                save_json(BOOKS_FILE, books)
        elif choice == 4:
            ok, msg = return_book(books, get_text_input("Book ID: "), user["username"])
            print(msg)
            if ok:
                save_json(BOOKS_FILE, books)
        elif choice == 5:
            my_books = [b for b in books if user["username"] in b["issued_to"]]
            display_books(my_books)
        elif choice == 6:
            print("Logged out.")
            return


def search_menu(books):
    """Ask what to search by, then show results."""
    print("Search by: 1. Title  2. Author  3. Book ID")
    option = get_int_input("Choose (1-3): ", 1, 3)
    fields = {1: "title", 2: "author", 3: "book_id"}
    results = search_record(books, get_text_input("Keyword: "), fields[option])
    print(f"{len(results)} result(s) found.")
    display_books(results)


def main():
    """Start the portal: load data, then keep running until the user exits."""
    print("=" * 40)
    print("   WELCOME TO THE LIBRARY PORTAL")
    print("=" * 40)
    books = load_json(BOOKS_FILE, [])
    users = load_json(USERS_FILE, [])
    if len(users) == 0:
        users = [{"username": "admin", "password": "admin123",
                  "name": "Librarian", "role": "admin"}]
        save_json(USERS_FILE, users)
        print("Created default admin account (admin / admin123).")

    while True:
        print("\n--- MAIN MENU ---\n1. Login\n2. Exit")
        choice = get_int_input("Choose (1-2): ", 1, 2)
        if choice == 1:
            user = login_screen(users)
            if user is None:
                continue
            if user["role"] == "admin":
                admin_menu(books, user)
            else:
                member_menu(books, user)
        else:
            save_json(BOOKS_FILE, books)
            print("Data saved. Goodbye!")
            break

---
# Sample Test Runs
The cells below test each function directly so the outputs can be seen without typing into the menu.

### Test 1: Create the sample data files
This writes fresh sample data, so the notebook gives the same results every time it is run.

In [9]:
sample_books = [
    {"book_id": "B001", "title": "Python Crash Course", "author": "Eric Matthes", "year": 2019, "total_copies": 3, "available": 3, "issued_to": []},
    {"book_id": "B002", "title": "Clean Code", "author": "Robert C. Martin", "year": 2008, "total_copies": 2, "available": 2, "issued_to": []},
    {"book_id": "B003", "title": "Introduction to Algorithms", "author": "Thomas H. Cormen", "year": 2009, "total_copies": 1, "available": 1, "issued_to": []},
    {"book_id": "B004", "title": "Automate the Boring Stuff", "author": "Al Sweigart", "year": 2020, "total_copies": 2, "available": 2, "issued_to": []},
    {"book_id": "B005", "title": "The Pragmatic Programmer", "author": "David Thomas", "year": 2019, "total_copies": 1, "available": 1, "issued_to": []}
]
sample_users = [
    {"username": "admin", "password": "admin123", "name": "Librarian", "role": "admin"},
    {"username": "ali", "password": "ali123", "name": "Ali Khan", "role": "member"},
    {"username": "sara", "password": "sara123", "name": "Sara Ahmed", "role": "member"}
]
print("Books saved:", save_json(BOOKS_FILE, sample_books))
print("Users saved:", save_json(USERS_FILE, sample_users))
if os.path.exists(LOG_FILE):
    os.remove(LOG_FILE)   # start with an empty transaction log
    print("Old transaction log cleared.")

Books saved: True
Users saved: True


### Test 2: Load data on startup (and handle a missing file)

In [10]:
books = load_json(BOOKS_FILE, [])
users = load_json(USERS_FILE, [])

# A file that does not exist: FileNotFoundError is caught, no crash
missing = load_json("data/does_not_exist.json", [])
print("Missing file returned:", missing)

[OK] Loaded 5 records from data/books.json
[OK] Loaded 3 records from data/users.json
[WARNING] data/does_not_exist.json not found. Starting with empty data.
Missing file returned: []


### Test 3: Login (correct and wrong password)

In [11]:
user = user_login(users, "ali", "ali123")
print("Correct login  ->", user["name"], "| role:", user["role"])

user = user_login(users, "ali", "wrongpass")
print("Wrong password ->", user)

Correct login  -> Ali Khan | role: member
Wrong password -> None


### Test 4: Add books (valid and invalid input)

In [12]:
print(add_record(books, "B006", "Data Science from Scratch", "Joel Grus", "2019", "2"))
print(add_record(books, "B007", "Bad Book", "Someone", "twenty", "2"))   # ValueError handled
print(add_record(books, "B001", "Duplicate", "Someone", "2020", "1"))   # duplicate ID
print(add_record(books, "B008", "", "Someone", "2020", "1"))            # empty title
print(add_record(books, "B009", "Future Book", "Someone", "2099", "1")) # invalid year
save_json(BOOKS_FILE, books)
display_books(books)

(True, "Book 'Data Science from Scratch' added successfully.")
(False, 'Year and copies must be whole numbers.')
(False, 'A book with ID B001 already exists.')
(False, 'Book ID, title and author cannot be empty.')
(False, 'Please enter a valid publication year.')
ID     Title                            Author                 Year   Avail 
----------------------------------------------------------------------------
B001   Python Crash Course              Eric Matthes           2019   3/3
B002   Clean Code                       Robert C. Martin       2008   2/2
B003   Introduction to Algorithms       Thomas H. Cormen       2009   1/1
B004   Automate the Boring Stuff        Al Sweigart            2020   2/2
B005   The Pragmatic Programmer         David Thomas           2019   1/1
B006   Data Science from Scratch        Joel Grus              2019   2/2


### Test 5: Search books

In [13]:
print("Title contains 'python':")
display_books(search_record(books, "python", "title"))
print("\nAuthor contains 'martin':")
display_books(search_record(books, "martin", "author"))
print("\nTitle contains 'java':")
display_books(search_record(books, "java", "title"))

Title contains 'python':
ID     Title                            Author                 Year   Avail 
----------------------------------------------------------------------------
B001   Python Crash Course              Eric Matthes           2019   3/3

Author contains 'martin':
ID     Title                            Author                 Year   Avail 
----------------------------------------------------------------------------
B002   Clean Code                       Robert C. Martin       2008   2/2

Title contains 'java':
No books to show.


### Test 6: Issue and return books

In [14]:
print(issue_book(books, "B003", "ali"))
print(issue_book(books, "B003", "sara"))   # no copies left
print(issue_book(books, "B001", "sara"))
print(issue_book(books, "B001", "ali"))
print(issue_book(books, "B999", "ali"))    # book does not exist
print(return_book(books, "B002", "ali"))   # was never issued to ali
print(return_book(books, "B001", "ali"))
print(issue_book(books, "B001", "ali"))
save_json(BOOKS_FILE, books)

(True, "'Introduction to Algorithms' issued to ali.")
(False, "No copies of 'Introduction to Algorithms' are available.")
(True, "'Python Crash Course' issued to sara.")
(True, "'Python Crash Course' issued to ali.")
(False, 'Book not found.')
(False, 'This book was not issued to you.')
(True, "'Python Crash Course' returned by ali.")
(True, "'Python Crash Course' issued to ali.")


### Test 7: Delete a book

In [15]:
print(delete_record(books, "B003"))   # still issued, so it is refused
print(delete_record(books, "B006"))
save_json(BOOKS_FILE, books)

(False, 'Cannot delete: some copies are still issued.')
(True, "Book 'Data Science from Scratch' deleted.")


### Test 8: Generate report

In [16]:
report = generate_report(books)
print_report(report)


========== LIBRARY REPORT ==========
Total titles       : 5
Total copies       : 9
Copies issued      : 3
Copies available   : 6
Out of stock       : Introduction to Algorithms
Total transactions : 5
Most popular book  : Python Crash Course (3 issues)


### Test 9: Check the saved data files
This shows that the JSON file was updated and that the CSV log was appended.

In [17]:
print("----- data/transactions.csv -----")
with open(LOG_FILE) as f:
    print(f.read())

print("----- data/books.json (first 2 books) -----")
with open(BOOKS_FILE) as f:
    print(json.dumps(json.load(f)[:2], indent=2))

----- data/transactions.csv -----
date_time,action,book_id,username
2026-11-14 21:13,ISSUE,B003,ali
2026-11-14 21:13,ISSUE,B001,sara
2026-11-14 21:13,ISSUE,B001,ali
2026-11-14 21:13,RETURN,B001,ali
2026-11-14 21:13,ISSUE,B001,ali

----- data/books.json (first 2 books) -----
[
  {
    "book_id": "B001",
    "title": "Python Crash Course",
    "author": "Eric Matthes",
    "year": 2019,
    "total_copies": 3,
    "available": 1,
    "issued_to": [
      "sara",
      "ali"
    ]
  },
  {
    "book_id": "B002",
    "title": "Clean Code",
    "author": "Robert C. Martin",
    "year": 2008,
    "total_copies": 2,
    "available": 2,
    "issued_to": []
  }
]


### Test 10: Full menu run with automated input
The interactive portal normally waits for keyboard input. Here we temporarily replace `input()` with a list of answers so a complete session runs in the notebook. The session includes wrong inputs (letters instead of numbers, a wrong password) to show that the portal handles them without crashing.

In [18]:
import builtins

answers = iter([
    "1", "admin", "wrong", "admin", "admin123",   # login: 1 failed attempt, then success
    "abc", "9",                                   # invalid menu choices
    "5",                                          # report
    "6",                                          # logout
    "1", "sara", "sara123", "5", "6",             # member views her books, logs out
    "2"                                           # exit
])

real_input = builtins.input
def fake_input(prompt=""):
    value = next(answers)
    print(prompt + value)
    return value

builtins.input = fake_input
try:
    main()
finally:
    builtins.input = real_input   # always restore the real input()

   WELCOME TO THE LIBRARY PORTAL
[OK] Loaded 5 records from data/books.json
[OK] Loaded 3 records from data/users.json

--- MAIN MENU ---
1. Login
2. Exit
Choose (1-2): 1
Username: admin
Password: wrong
Wrong username or password. 2 attempt(s) left.
Username: admin
Password: admin123

Welcome, Librarian! (role: admin)

--- ADMIN MENU ---
1. View all books
2. Add book
3. Search books
4. Delete book
5. Generate report
6. Logout
Choose (1-6): abc
Invalid input. Please type a number.
Choose (1-6): 9
Please enter a number from 1 to 6.
Choose (1-6): 5

========== LIBRARY REPORT ==========
Total titles       : 5
Total copies       : 9
Copies issued      : 3
Copies available   : 6
Out of stock       : Introduction to Algorithms
Total transactions : 5
Most popular book  : Python Crash Course (3 issues)

--- ADMIN MENU ---
1. View all books
2. Add book
3. Search books
4. Delete book
5. Generate report
6. Logout
Choose (1-6): 6
Logged out.

--- MAIN MENU ---
1. Login
2. Exit
Choose (1-2): 1
Usern

### Run the portal yourself
Uncomment the line below and run the cell to use the portal interactively.

Logins: `admin / admin123`, `ali / ali123`, `sara / sara123`

In [19]:
# main()